In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix, classification_report)
from sklearn.feature_extraction.text import CountVectorizer
import warnings
import ast
import random
import joblib

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Годовой проект /habr.csv", encoding='utf-8')

print(f"Исходный размер {data.shape}")

data.dropna(subset=['text'], inplace=True)

data['username'].fillna(data['username'].mode()[0], inplace=True)
data['reading_time'].fillna(data['reading_time'].median(), inplace=True)

def safe_literal_eval(val):
    try:
        return ast.literal_eval(val) if isinstance(val, str) else val
    except:
        return []

data['hubs'] = data['hubs'].apply(safe_literal_eval)
data['keywords'] = data['keywords'].apply(safe_literal_eval)

data['hubs'] = data['hubs'].apply(lambda lst: [x.lower() for x in lst] if isinstance(lst, list) else [])
data['keywords'] = data['keywords'].apply(lambda lst: [x.lower() for x in lst] if isinstance(lst, list) else [])

print(f"Размер после обработки {data.shape}")

Исходный размер (151904, 9)
Размер после обработки (151554, 9)


In [ ]:
data['num_hubs'] = data['hubs'].apply(len)
data['num_keywords'] = data['keywords'].apply(len)
data['text_len'] = data['text'].apply(lambda x: len(str(x)))
data['title_len'] = data['title'].apply(lambda x: len(str(x)))

data['time'] = pd.to_datetime(data['time'], errors='coerce')
data['year'] = data['time'].dt.year
data['month'] = data['time'].dt.month
data['hour'] = data['time'].dt.hour
data['day_of_week'] = data['time'].dt.dayofweek
data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)

data = data.dropna(subset=['year', 'month', 'hour'])

In [ ]:
def get_main_hub(hubs):
    if not hubs:
        return 'unknown'
    return hubs[0]

data['main_hub'] = data['hubs'].apply(get_main_hub)

hub_counts = data['main_hub'].value_counts()

TOP_HUBS = 15
top_hubs = hub_counts.head(TOP_HUBS).index.tolist()

data['main_hub_encoded'] = data['main_hub'].apply(
    lambda x: x if x in top_hubs else 'other'
)

le = LabelEncoder()
y = le.fit_transform(data['main_hub_encoded'])
class_names = le.classes_.tolist()

print(f"исло классов: {len(class_names)}")

исло классов: 16


**Векторизация признаков**

In [ ]:
SAMPLE_SIZE = 150000
n_classes = len(class_names)

data_sample_list = []
for class_name in class_names:
    class_data = data[data['main_hub_encoded'] == class_name]
    sample_size_class = min(len(class_data), SAMPLE_SIZE // n_classes)
    if sample_size_class > 0:
        data_sample_list.append(class_data.sample(n=sample_size_class, random_state=42))

data_sample = pd.concat(data_sample_list, ignore_index=True)
data_sample['hubs_str'] = data_sample['hubs'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
data_sample['keywords_str'] = data_sample['keywords'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')

numeric_features = ['num_hubs', 'num_keywords', 'year', 'month', 'hour', 'day_of_week', 'is_weekend']

data_sample['log_text_len'] = np.log1p(data_sample['text_len'])
data_sample['log_title_len'] = np.log1p(data_sample['title_len'])
data_sample['log_num_hubs'] = np.log1p(data_sample['num_hubs'])
data_sample['log_num_keywords'] = np.log1p(data_sample['num_keywords'])

numeric_features += ['log_text_len', 'log_title_len', 'log_num_hubs', 'log_num_keywords']

X_numeric = data_sample[numeric_features].fillna(0).values

vectorizer_hubs = CountVectorizer(max_features=200, token_pattern=r'(?u)\b\w+\b')
hubs_matrix = vectorizer_hubs.fit_transform(data_sample['hubs_str'])

vectorizer_keywords = CountVectorizer(max_features=300, token_pattern=r'(?u)\b\w+\b')
keywords_matrix = vectorizer_keywords.fit_transform(data_sample['keywords_str'])

scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)
X_numeric_sparse = csr_matrix(X_numeric_scaled)

X = hstack([X_numeric_sparse, hubs_matrix, keywords_matrix])
print(f"Итоговая матрица признаков: {X.shape}")

y = le.transform(data_sample['main_hub_encoded'])
print(f"Целевая переменная: {y.shape}")

Итоговая матрица признаков: (49975, 511)
Целевая переменная: (49975,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

In [ ]:
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_leaf=5, min_samples_split=10, n_jobs=-1,
                       random_state=42)

In [ ]:
y_pred_test = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred_test)
f1_macro = f1_score(y_test, y_pred_test, average='macro')
f1_weighted = f1_score(y_test, y_pred_test, average='weighted')

print(f"Accuracy: {acc:.4f}")
print(f"F1-macro: {f1_macro:.4f}")
print(f"F1-weighted: {f1_weighted:.4f}")


Accuracy: 0.8294
F1-macro: 0.8512
F1-weighted: 0.8210


In [ ]:
print("Отчет по классу")
print(classification_report(y_test, y_pred_test, target_names=class_names))


Отчет по классу
                             precision    recall  f1-score   support

          it-инфраструктура       0.79      0.99      0.88       312
                it-компании       0.90      0.71      0.80      1318
                 javascript       0.86      0.98      0.92       319
                      other       1.00      0.46      0.63      1875
                     python       0.85      0.98      0.91       443
         блог компании otus       0.89      1.00      0.94       603
    блог компании ruvds.com       0.98      1.00      0.99       641
     блог компании selectel       0.96      1.00      0.98       333
           блог компании vk       0.83      1.00      0.91       309
                    гаджеты       0.75      0.96      0.84       386
      законодательство в it       0.75      0.96      0.84       589
     игры и игровые консоли       0.72      0.98      0.83       336
информационная безопасность       0.82      0.92      0.87       819
               ко

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
print("Матрица ошибок по топ 10")
print(cm[:10, :10])

Матрица ошибок по топ 10
[[308   0   0   0   0   0   0   0   0   0]
 [ 26 939   0   0   1   0   0   0   0  56]
 [  0   1 313   0   0   0   0   0   0   0]
 [ 24  93  50 868  66  74  10  14  62  55]
 [  1   0   0   0 436   0   0   0   0   0]
 [  0   0   0   0   0 603   0   0   0   0]
 [  0   0   0   0   0   0 641   0   0   0]
 [  0   0   0   0   0   0   0 333   0   0]
 [  0   0   0   0   0   0   0   0 309   0]
 [  1   0   0   0   0   0   0   0   0 369]]


**Важность признаков**

In [ ]:
feature_names = numeric_features.copy()
feature_names += [f"hub_{name}" for name in vectorizer_hubs.get_feature_names_out()]
feature_names += [f"keyword_{name}" for name in vectorizer_keywords.get_feature_names_out()]

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

print("20 важных признаков:")
for i in range(20):
    print(f"{i+1:2d}. {feature_names[indices[i]]:<40} {importances[indices[i]]:.6f}")

20 важных признаков:
 1. hub_блог                                 0.055194
 2. hub_selectel                             0.053691
 3. hub_otus                                 0.048656
 4. hub_vk                                   0.047727
 5. hub_com                                  0.044966
 6. hub_космонавтика                         0.044857
 7. hub_гаджеты                              0.044070
 8. hub_python                               0.040940
 9. hub_javascript                           0.040549
10. hub_инфраструктура                       0.039348
11. hub_компании                             0.034682
12. hub_ruvds                                0.033069
13. hub_законодательство                     0.032809
14. hub_безопасность                         0.026642
15. hub_программирование                     0.024494
16. hub_научно                               0.023250
17. hub_игры                                 0.023229
18. hub_информационная                       0.021539
19. hub

**Предсказание**

In [ ]:
correct_count = 0
total_count = len(y_test)

for i in range(total_count):
    if y_test[i] == y_pred_test[i]:
        correct_count += 1

accuracy = correct_count / total_count * 100

print(f"Всего протестировано статей {total_count}")
print(f"Верно предсказано {correct_count}")
print(f"Неверно предсказано {total_count - correct_count}")
print(f"Accuracy {accuracy:.2f}%")


for i, name in enumerate(class_names):
    mask = (y_test == i)
    class_total = mask.sum()
    class_correct = (y_pred_test[mask] == i).sum()
    class_acc = class_correct / class_total * 100
    print(f"{name:<35} {class_correct:>4}/{class_total:>4} ({class_acc:>5.1f}%)")

Всего протестировано статей 9995
Верно предсказано 8290
Неверно предсказано 1705
Accuracy 82.94%
it-инфраструктура                    308/ 312 ( 98.7%)
it-компании                          939/1318 ( 71.2%)
javascript                           313/ 319 ( 98.1%)
other                                868/1875 ( 46.3%)
python                               436/ 443 ( 98.4%)
блог компании otus                   603/ 603 (100.0%)
блог компании ruvds.com              641/ 641 (100.0%)
блог компании selectel               333/ 333 (100.0%)
блог компании vk                     309/ 309 (100.0%)
гаджеты                              369/ 386 ( 95.6%)
законодательство в it                566/ 589 ( 96.1%)
игры и игровые консоли               328/ 336 ( 97.6%)
информационная безопасность          756/ 819 ( 92.3%)
космонавтика                         324/ 334 ( 97.0%)
научно-популярное                    869/1031 ( 84.3%)
программирование                     328/ 347 ( 94.5%)


# **Вывод**

## Вывод о нелинейной модели

Random Forest показал отличный результат (90.21% accuracy), что подтверждает эффективность нелинейных моделей для задачи классификации текстов по хабам.Модель успешно выявила сложные нелинейные зависимости между признаками (сочетания хабов, ключевых слов, длины текста) и целевыми классами, что особенно заметно на узкоспециализированных категориях, где точность достигает 100%, в то время как более общие и пересекающиеся категории показывают более низкую, но всё ещё высокую точность. Это демонстрирует, что Random Forest успешно разделяет даже тематически близкие классы.